[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abhiksark/pycon-italy-2026-workshop/blob/main/notebooks/05-tiled-matmul-triton.ipynb)

# 05 · Tiled matmul in Triton - _the capstone_

We compute `C = A @ B` where A is `(M, K)` and B is `(K, N)`. Each program computes one `(BLOCK_M, BLOCK_N)` tile of C by streaming through K in chunks of `BLOCK_K`.

**Six TODOs.** When the last one is filled in and the correctness check is green, you have written your first high-performance GPU kernel in Python.

## Environment bootstrap

In [ ]:
import importlib, subprocess, sys
def ensure(pip_name, import_name=None):
    try: importlib.import_module(import_name or pip_name)
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pip_name])
ensure('triton')
import torch, triton
import triton.language as tl
assert torch.cuda.is_available(), 'No GPU detected.'
print(f'triton {triton.__version__} | gpu {torch.cuda.get_device_name(0)}')

In [ ]:
# Colab path bootstrap: if we are not at the repo root, clone the repo and chdir.
import os, subprocess, pathlib
if not pathlib.Path('notebooks/shared').is_dir():
    repo_url = os.environ.get('WORKSHOP_REPO_URL', 'https://github.com/abhiksark/pycon-italy-2026-workshop.git')
    target = '/content/pycon-italy'
    if not pathlib.Path(target).exists():
        subprocess.check_call(['git', 'clone', '--depth=1', repo_url, target])
    os.chdir(target)
print(f'cwd: {os.getcwd()}')

## Inputs and reference output

In [ ]:
M, N, K = 512, 512, 512  # small enough to iterate fast on a T4
A = torch.randn((M, K), device='cuda', dtype=torch.float32)
B = torch.randn((K, N), device='cuda', dtype=torch.float32)
reference = A @ B
print(f'A: {tuple(A.shape)} | B: {tuple(B.shape)} | C: ({M}, {N})')

## The kernel

Read the comments above every TODO. The structure of the kernel is given; you fill in the meaningful lines.

nb04's blur assumed a contiguous tensor and indexed with a bare `* W`. Real tensors carry **strides** - the element gap between consecutive rows and columns - so this kernel takes them explicitly (`stride_am`, `stride_ak`, ...) and never assumes a memory layout.

**Mental model:** program `(pid_m, pid_n)` is responsible for the `(BLOCK_M, BLOCK_N)` output tile at row-block `pid_m`, col-block `pid_n`. To compute that tile we accumulate `K // BLOCK_K` partial products of `(BLOCK_M, BLOCK_K) @ (BLOCK_K, BLOCK_N)` tiles.

![diagram](https://raw.githubusercontent.com/abhiksark/pycon-italy-2026-workshop/main/notebooks/shared/diagrams/nb05-kloop.png)

Mental picture: the K-loop sweeps a row of A and a column of B, accumulating their products into one tile of C.

![diagram](https://raw.githubusercontent.com/abhiksark/pycon-italy-2026-workshop/main/notebooks/shared/diagrams/nb05-matmul-tiling.png)

In [ ]:
@triton.jit
def matmul_kernel(
    a_ptr, b_ptr, c_ptr,
    M, N, K,
    stride_am, stride_ak,
    stride_bk, stride_bn,
    stride_cm, stride_cn,
    BLOCK_M: tl.constexpr, BLOCK_N: tl.constexpr, BLOCK_K: tl.constexpr,
):
    # TODO (1/6): get pid_m and pid_n from program_id axes 0 and 1.
    # pid_m = ...
    # pid_n = ...

    # TODO (2/6): row and column block offsets within A and B.
    # offs_m = pid_m * BLOCK_M + tl.arange(0, BLOCK_M)   # rows of A and C this tile owns
    # offs_n = pid_n * BLOCK_N + tl.arange(0, BLOCK_N)   # cols of B and C this tile owns
    # offs_k = tl.arange(0, BLOCK_K)                       # K-tile offsets (re-stepped each iteration)

    # TODO (3/6): initialise the accumulator tile to zero, shape (BLOCK_M, BLOCK_N), dtype float32.
    # acc = tl.zeros((BLOCK_M, BLOCK_N), dtype=tl.float32)

    # TODO (4/6): the K-loop. Iterate over K in chunks of BLOCK_K. Each iteration:
    #   - compute A-tile pointers a_ptrs of shape (BLOCK_M, BLOCK_K)
    #   - compute B-tile pointers b_ptrs of shape (BLOCK_K, BLOCK_N)
    #   - mask out-of-range K-loads (only matters in the last K-block when K % BLOCK_K != 0)
    #   - tl.load A and B with masks, other=0.0
    #   - acc += tl.dot(a, b)
    #   - advance offs_k by BLOCK_K
    # for k_start in range(0, K, BLOCK_K):
    #     ...

    # TODO (5/6): output offsets and a 2D mask over the (BLOCK_M, BLOCK_N) tile.
    # c_ptrs = c_ptr + offs_m[:, None] * stride_cm + offs_n[None, :] * stride_cn
    # c_mask = (offs_m[:, None] < M) & (offs_n[None, :] < N)

    # TODO (6/6): store the accumulator. Cast to the output dtype if needed.
    # tl.store(c_ptrs, acc, mask=c_mask)

    pass

![diagram](https://raw.githubusercontent.com/abhiksark/pycon-italy-2026-workshop/main/notebooks/shared/diagrams/nb05-store.png)

## Launch and correctness

In [ ]:
BLOCK_M = BLOCK_N = BLOCK_K = 64
C = torch.empty((M, N), device='cuda', dtype=torch.float32)

grid = (triton.cdiv(M, BLOCK_M), triton.cdiv(N, BLOCK_N))
matmul_kernel[grid](
    A, B, C,
    M, N, K,
    A.stride(0), A.stride(1),
    B.stride(0), B.stride(1),
    C.stride(0), C.stride(1),
    BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_K=BLOCK_K,
)
torch.testing.assert_close(C, reference, rtol=1e-3, atol=1e-3)
print('correctness: ok')

## Benchmark

In [ ]:
import sys, pathlib
# Make `from notebooks.shared.*` resolve whether Jupyter was launched
# from the repo root, from notebooks/, or anywhere below.
for _root in (pathlib.Path.cwd(), *pathlib.Path.cwd().parents):
    if (_root / 'notebooks' / 'shared' / 'benchmark_utils.py').is_file():
        if str(_root) not in sys.path:
            sys.path.insert(0, str(_root))
        break
from notebooks.shared.benchmark_utils import gflops_for_matmul, median_seconds

def run_triton():
    matmul_kernel[grid](
        A, B, C, M, N, K,
        A.stride(0), A.stride(1),
        B.stride(0), B.stride(1),
        C.stride(0), C.stride(1),
        BLOCK_M=BLOCK_M, BLOCK_N=BLOCK_N, BLOCK_K=BLOCK_K,
    )
    torch.cuda.synchronize()

def run_torch():
    torch.matmul(A, B, out=C)
    torch.cuda.synchronize()

t_triton = median_seconds(run_triton, runs=20, warmup=5)
t_torch = median_seconds(run_torch, runs=20, warmup=5)
g_triton = gflops_for_matmul(M, N, K, t_triton)
g_torch = gflops_for_matmul(M, N, K, t_torch)
print(f'triton: {t_triton*1e3:7.3f} ms | {g_triton:8.1f} GFLOPs')
print(f'torch:  {t_torch*1e3:7.3f} ms | {g_torch:8.1f} GFLOPs')
print(f'ratio:  {g_triton / g_torch * 100:5.1f}% of cuBLAS')

# --- where on the roofline are we? ---
T4_PEAK_TFLOPS_FP32 = 8.1     # T4 datasheet, fp32, no tensor cores
T4_PEAK_HBM_GBS = 320.0       # T4 datasheet
machine_balance = T4_PEAK_TFLOPS_FP32 * 1e3 / T4_PEAK_HBM_GBS  # FLOPs/byte at the knee
ai = (2 * M * N * K) / (4 * (M*K + K*N + M*N))  # ideal AI: each matrix loaded once
peak_pct = (g_triton / 1e3) / T4_PEAK_TFLOPS_FP32 * 100
print(f'arithmetic intensity: {ai:.0f} FLOPs/byte (compute-bound > {machine_balance:.0f})')
print(f'achieved:             {g_triton/1e3:.2f} TFLOPS ({peak_pct:.0f}% of T4 peak fp32)')


## You did it

If `correctness: ok` printed and the ratio is somewhere in the 30–60% range on a T4, you have a working tiled matmul. The remaining 40–70% gap to `torch.matmul` is what autotuning, Tensor Cores, and `tl.dot` precision modes buy you - each one is a workshop of its own.

Run `bench/compare_vs_torch.py` to see the same kernel against a naive Python triple-loop and `numpy.matmul`. The naive-Python baseline is _five orders of magnitude_ slower than what you just wrote.

## Going further

This kernel is deliberately the *simple* tiled matmul. Four best-practice levers carry it toward `torch.matmul` - each is one line of intent and a chapter of depth:

| Lever | What it does | Typical win |
|-------|--------------|-------------|
| `@triton.autotune` | Compiles several `BLOCK_*` / `num_warps` / `num_stages` configs, benchmarks them on the first call, caches the winner per shape | the best tile for *your* GPU |
| `num_stages` pipelining | Overlaps the next K-tile's HBM load with the current tile's `tl.dot` - software pipelining | hides memory latency |
| `GROUP_SIZE_M` swizzling | Re-orders program IDs so neighbouring output tiles reuse the A/B rows already sitting in L2 | ~1.3x, +60% L2 hit rate |
| Tensor Cores (TF32) | Drop `input_precision='ieee'` and Ampere+ runs `tl.dot` on TF32 tensor cores | several x - at reduced precision |

The official [Triton matmul tutorial](https://triton-lang.org/main/getting-started/tutorials/03-matrix-multiplication.html) layers all four onto exactly this kernel. Start with `@triton.autotune` - highest reward for the least code.